# Baseline Model

## Background & Context

Our baseline model says that we have power plants $i$ and towns $j$ and a power line connecting each power plant to each town. The cost of this model depends on the cost of transmission across distance.

In real-world electricity markets managed by entities like ISO-New England, pushing power across the grid incurs "wheeling tariffs" and congestion charges. These compensate for physical power dissipated as heat ($I^2R$ ohmic losses) over High-Voltage Alternating Current (HVAC) lines and the amortized operational cost of the infrastructure (Bialek, J., 1996. *Tracing the flow of electricity. IEE Proceedings-Generation, Transmission and Distribution*). To model this transmission friction, we assign a standard proxy transmission fee of $0.05 per kilometer per Megawatt (MWh).

We aim to minimize the total combined cost of generation and transmission while strictly meeting demand constraints. In this initial baseline model, we assume infinite generation capacity, infinite line capacity, and set production costs to $0$. This isolates the pure geographic routing penalty.

## Formulation

To model the total generation capacity, we introduce a "source" node $0$ that provides the exact network demand to all power plants for free.

Each line is indexed as $(i,j) \in L$ where $i$ is the power plant (or the source), and $j$ is the town (or power plant). The capacity (upper bound) for each line is infinite and the least amount of power allowed through each line (lower bound) is $0$:
$$u_{ij} = \infty 
\\ l_{ij} = 0$$

The distance between power plant and town (length of the transmission line) is denoted by:
$$d_{ij}$$

Cost per unit of power flowing through the transmission line in dollars (cost from source to power plants is $0$ in the baseline):
$$c_{ij} = 0.05 \times d_{ij}$$

Supply from source $0$:
$$b(0) = -\sum_{j} D_j$$

Nodes for power plants $i$ act as transshipment nodes:
$$b(i) = 0$$

Demand from town $j$:
$$b(j) = D_j \ge 0$$

Amount of power flowing through the transmission line:
$$x_{ij} \ge 0$$

### Objective Function
$$\min \sum_{(i,j) \in L} c_{ij} x_{ij}$$

### Constraints
Aside from the upper/lower bound and non-negative constraint from above, we have the balance constraints (flow out minus flow in equals supply/demand at that node):
$$\sum_{j: (i,j) \in L} x_{ij} - \sum_{k: (k,i) \in L} x_{ki} = -b(i)$$
*(Note: Following Gurobi min-flow solver conventions, demand is strictly positive and supply is strictly negative).*

In [13]:
from gurobi_optimods.min_cost_flow import min_cost_flow_pandas
import pandas as pd

plants_df = pd.read_csv("Distances - Power Plants.csv")
towns_df = pd.read_csv("Distances - Towns.csv")
pairs_df = pd.read_csv("distance_pairs.csv")

num_plants = len(plants_df)
total_demand = towns_df["Demand"].sum()

# Node data
# We add a source (node 0) with supply = total_demand
# The plants (1-5) will have 0 demand, and the towns will have positive demand.
node_data = pd.DataFrame({
    "demand": [-total_demand] + [0] * num_plants + towns_df["Demand"].to_list()
}, index=range(0, 16)) # 0: source, 1-5: plants, 6-15: towns

# Edge data
edge_data = pairs_df[["power_plant_id", "town_id", "distance"]].copy()
edge_data["source"] = edge_data["power_plant_id"]
edge_data["target"] = edge_data["town_id"] + num_plants
edge_data["cost"] = 0.05 * edge_data["distance"]
edge_data["capacity"] = float("inf")
edge_data = edge_data[["source", "target", "cost", "capacity"]]

# Create edges from source to each plant with 0 cost
source_edges = pd.DataFrame({
    "source": [0] * num_plants,
    "target": range(1, num_plants + 1),
    "cost": [0.0] * num_plants,
    "capacity": [float("inf")] * num_plants
})

edge_data = pd.concat([source_edges, edge_data]).set_index(["source", "target"])

In [4]:
edge_data, node_data

(                   cost  capacity
 source target                    
 0      1       0.000000       inf
        2       0.000000       inf
        3       0.000000       inf
        4       0.000000       inf
        5       0.000000       inf
 1      6       0.002709       inf
        7       0.003292       inf
        8       0.003304       inf
        9       0.000246       inf
        10      0.001309       inf
        11      0.002386       inf
        12      0.006065       inf
        13      0.006854       inf
        14      0.003574       inf
        15      0.007736       inf
 2      6       0.000498       inf
        7       0.000820       inf
        8       0.000856       inf
        9       0.002847       inf
        10      0.002852       inf
        11      0.001126       inf
        12      0.003487       inf
        13      0.005633       inf
        14      0.003444       inf
        15      0.008729       inf
 3      6       0.007702       inf
        7       0.00

In [14]:
obj, sol = min_cost_flow_pandas(edge_data, node_data, verbose=False)

In [15]:
print(f"Total Cost: {obj}")
sol[sol > 0]

Total Cost: 3199.920729854883


source  target
0       1          500.0
        2         1650.0
        3          230.0
        5          270.0
1       9          250.0
        10         250.0
2       6          650.0
        7          450.0
        8          350.0
        11         200.0
3       13         150.0
        15          80.0
5       12         150.0
        14         120.0
Name: flow, dtype: float64

# Augmented Model

## Power Plant Costs & Generation Constraints (Economic Dispatch)

In reality, power plants do not have infinite capacity, and their production costs vary wildly depending on fuel type. To align with real-world short-run marginal costs (often modeled in the US Energy Information Administration's *Levelized Cost of Energy* (LCOE) reports):
1. Baseload Plants (Hydro & Nuclear):*High capital costs to build, but extremely low operational/fuel costs ($15 - $25/MWh). They form the backbone of the grid.
2. Intermediate/Peaking Plants (Natural Gas): Medium fuel costs ($45 - $50/MWh). Turned on when demand exceeds baseload.
3. Expensive/Decommissioning (Coal): High operating costs + emission drawbacks ($85/MWh). Used only as a last resort.

We'll include now the cost of electricity from the arcs from the source node to the power plants. We also include an upper bound capacity for how much electricity each plant can produce.

In [16]:
# We copy the setup from before but update source edges
aug_edge_data = pairs_df[["power_plant_id", "town_id", "distance"]].copy()
aug_edge_data["source"] = aug_edge_data["power_plant_id"]
aug_edge_data["target"] = aug_edge_data["town_id"] + num_plants
aug_edge_data["cost"] = 0.05 * aug_edge_data["distance"]
aug_edge_data["capacity"] = float("inf")
aug_edge_data = aug_edge_data[["source", "target", "cost", "capacity"]]

# Source edges now use actual plant constraints
# Cost = production cost, Capacity = generation capacity
aug_source_edges = pd.DataFrame({
    "source": [0] * num_plants,
    "target": range(1, num_plants + 1),
    "cost": plants_df["Cost"].tolist(),
    "capacity": plants_df["Capacity"].tolist()
})

aug_edge_data = pd.concat([aug_source_edges, aug_edge_data]).set_index(["source", "target"])

# Node data remains exactly the same as the baseline
aug_obj, aug_sol = min_cost_flow_pandas(aug_edge_data, node_data, verbose=False)

print(f"Total Augmented Cost: {aug_obj}")
aug_sol[aug_sol > 0]

Total Augmented Cost: 90281.340424487


source  target
0       1          299.0
        2          745.0
        3          360.0
        4         1246.0
1       9           49.0
        10         250.0
2       6          605.0
        12         140.0
3       12          10.0
        13         150.0
        14         120.0
        15          80.0
4       6           45.0
        7          450.0
        8          350.0
        9          201.0
        11         200.0
Name: flow, dtype: float64

## Transmission Line Capacities

In reality, transmission lines do not have infinite capacity. High-voltage lines are subject to strict thermal limits; pushing too much power through them causes the physical cables to heat up, sag, or even melt.

For context, according to the [US Department of Energy's National Transmission Grid Study](https://www.energy.gov/sites/prod/files/oeprod/DocumentsandMedia/TransmissionGrid.pdf), a standard 230 kilovolt (kV) High-Voltage Alternating Current (HVAC) transmission line typically has a thermal capacity of roughly **400 MW**.

To make our model more realistic, rather than letting a single super-efficient line carry an infinite amount of power to a large city, we will cap the capacity of every single line from a power plant to a town at $400$ MW. If a town's demand exceeds the cheapest direct route, the solver will be forced to route the remaining power through secondary, more distant pathways.

In [17]:
# Augmented Model with Transmission Line Capacities
cap_edge_data = pairs_df[["power_plant_id", "town_id", "distance"]].copy()
cap_edge_data["source"] = cap_edge_data["power_plant_id"]
cap_edge_data["target"] = cap_edge_data["town_id"] + num_plants

# Transmission cost ($0.05 per km per MW)
cap_edge_data["cost"] = 0.05 * cap_edge_data["distance"]

# Add a transmission limit of 400 MW per line
cap_edge_data["capacity"] = 400.0 
cap_edge_data = cap_edge_data[["source", "target", "cost", "capacity"]]

# Super source edges (Plant generation constraints)
cap_super_source_edges = pd.DataFrame({
    "source": [0] * num_plants,
    "target": range(1, num_plants + 1),
    "cost": plants_df["Cost"].tolist(),
    "capacity": plants_df["Capacity"].tolist()  # Plant limits remain the same
})

cap_edge_data = pd.concat([cap_super_source_edges, cap_edge_data]).set_index(["source", "target"])

# Solve
cap_obj, cap_sol = min_cost_flow_pandas(cap_edge_data, node_data, verbose=False)

print(f"Total Cost with Line Capacities & Realistic Transmission Cost: {cap_obj}")
cap_sol[cap_sol > 0]

Total Cost with Line Capacities & Realistic Transmission Cost: 90329.40428371924


source  target
0       1          299.0
        2          745.0
        3          360.0
        4         1246.0
1       9           49.0
        10         250.0
2       6          400.0
        7          205.0
        12         140.0
3       12          10.0
        13         150.0
        14         120.0
        15          80.0
4       6          250.0
        7          245.0
        8          350.0
        9          201.0
        11         200.0
Name: flow, dtype: float64

# Interpretation & Real-World Impact

### System Behavior Under Constraints
By observing the solver outputs progressively, we see realistic grid behaviors organically emerge:
1. The cheapest baseload energy (Nuclear: Seabrook, 1246 MW and Hydroelectric: Comerford, 360 MW) is maximized immediately to 100% capacity. Natural Gas plants are utilized dynamically to bridge the final gaps in town demand.
2. Transmission Bottlenecks: When line thermal limits (400 MW) are enforced, large load centers like Manchester (650 MW demand) cannot be fed entirely by a single optimal line. The solver correctly splinters the power flow across multiple generating sources to bypass network congestion.

### Coal Phase-Out
The model output shows that flow from the Merrimack Station (Coal/Oil) strictly rests at $0$ MW in the optimal baseline. The solver completely ignores the coal plant because its marginal cost (\$85/MWh) is uncompetitive. 

This mathematical result mirrors reality. Due to cheaper natural gas and renewables driving out expensive coal operations, [Merrimack Station—New England's last remaining coal plant—officially shut down and ceased operations recently](https://www.nhpr.org/nh-news/2025-10-06/new-englands-last-coal-plant-has-stopped-operating-according-to-its-owners). Our network model simulates the real-world economic decommissioning of coal infrastructure.